# Notebook 02: XGBoost Baseline Model
## T2D Risk Predictor — Fairness-Aware ML for Healthcare
**PhD Research | University of Portsmouth | Kesser Karim**

### Objectives
1. Train a baseline XGBoost classifier on preprocessed NHANES features
2. Evaluate with 5-fold stratified cross-validation (AUROC, ECE, F1)
3. Generate calibration curves and SHAP feature importance
4. Establish baseline fairness metrics (equalized odds per subgroup)
5. Log experiment to MLflow for reproducibility

In [ ]:
# === SETUP ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import mlflow
import mlflow.xgboost
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from xgboost import XGBClassifier
from fairlearn.metrics import MetricFrame, equalized_odds_difference

DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_experiment('t2d-risk-predictor')

## 1. Load Preprocessed Features

In [ ]:
# Load features engineered from 01_eda_nhanes.ipynb
df = pd.read_parquet(DATA_DIR / 'nhanes_features.parquet')

# Feature set (clinical + demographic)
FEATURES = [
    'RIDAGEYR',   # Age
    'BMXBMI',     # BMI
    'BPXOSY1',    # Systolic BP
    'LBXTC',      # Total cholesterol
    'INDFMPIR',   # Poverty-income ratio
    'RIAGENDR',   # Sex
    'RIDRETH3',   # Ethnicity (encoded)
]

X = df[FEATURES]
y = df['t2d']
sensitive = df['ethnicity']  # for Fairlearn

print(f'Shape: {X.shape} | T2D prevalence: {y.mean():.3f}')
print(f'Subgroup counts:\n{sensitive.value_counts()}')

## 2. XGBoost Pipeline with 5-Fold CV

In [ ]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y==0).sum()/(y==1).sum(),  # handle imbalance
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False,
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_prob = cross_val_predict(pipeline, X, y, cv=cv, method='predict_proba')[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

auroc = roc_auc_score(y, y_prob)
print(f'5-Fold CV AUROC: {auroc:.4f}')
print(classification_report(y, y_pred, target_names=['No T2D', 'T2D']))

## 3. Fairness Audit (Equalized Odds per Ethnicity)

In [ ]:
from fairlearn.metrics import MetricFrame, equalized_odds_difference, true_positive_rate, false_positive_rate

mf = MetricFrame(
    metrics={
        'TPR': true_positive_rate,
        'FPR': false_positive_rate,
        'AUROC': roc_auc_score,
    },
    y_true=y,
    y_pred=y_pred,
    sensitive_features=sensitive,
)

print('Fairness metrics by subgroup:')
print(mf.by_group.round(3))

eod = equalized_odds_difference(y, y_pred, sensitive_features=sensitive)
print(f'\nEqualized Odds Difference (baseline): {eod:.4f}')
print('Target: < 0.10 after Fairlearn post-processing')

## 4. SHAP Feature Importance

In [ ]:
# Fit on full training set for SHAP
pipeline.fit(X, y)
clf = pipeline.named_steps['clf']
X_transformed = pipeline[:-1].transform(X)

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_transformed)

plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_transformed, feature_names=FEATURES,
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance — Baseline XGBoost', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_shap_importance.png', dpi=150)
plt.show()

## 5. MLflow Logging

Log all metrics and the model artifact for reproducibility and deployment tracking.

In [ ]:
with mlflow.start_run(run_name='xgboost-baseline'):
    mlflow.log_params({
        'n_estimators': 300, 'max_depth': 5,
        'learning_rate': 0.05, 'cv_folds': 5,
    })
    mlflow.log_metrics({
        'auroc': auroc,
        'equalized_odds_diff': eod,
    })
    mlflow.xgboost.log_model(clf, 'xgboost-baseline')
    print('Run logged to MLflow')

## 6. Results Summary

| Metric | Baseline Value | Target |
|--------|---------------|--------|
| AUROC (5-fold CV) | TBD | > 0.82 |
| Equalized Odds Diff | TBD | < 0.10 |
| F1 (T2D class) | TBD | > 0.70 |

**Next:** `03_fairness_audit.ipynb` — Fairlearn threshold optimiser + reweighing to reduce parity gap